# Chapter 25: Experimental Design and A/B Testing

**Level:** Analytical  
**Objectives:** design a randomized comparison; estimate continuous and binary effects; diagnose assignment; use permutation inference.  
**Prerequisites:** Chapters 19 to 22.  
**Estimated study time:** 75 minutes.

The data below are synthetic NRG distributor-account outcomes generated with seed 20260829.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260829)
n = 2000
account_id = np.arange(n)
region = rng.choice(["Java", "Sumatra", "Other"], n, p=[0.55, 0.25, 0.20])
treatment = np.zeros(n, dtype=int)
for value in np.unique(region):
    indices = np.flatnonzero(region == value)
    rng.shuffle(indices)
    treatment[indices[:len(indices) // 2]] = 1
baseline = np.where(region == "Java", 0.31, np.where(region == "Sumatra", 0.27, 0.24))
completed = rng.binomial(1, np.clip(baseline + 0.035 * treatment, 0, 1))
minutes = rng.normal(9.8 - 0.7 * treatment, 2.2, n)

int(treatment.sum()), np.bincount(treatment).tolist()

In [2]:
control_minutes = minutes[treatment == 0]
treatment_minutes = minutes[treatment == 1]
minute_effect = treatment_minutes.mean() - control_minutes.mean()
control_rate = completed[treatment == 0].mean()
treatment_rate = completed[treatment == 1].mean()
risk_difference = treatment_rate - control_rate
relative_lift = risk_difference / control_rate
print(f"Time effect: {minute_effect:.3f} minutes")
print(f"Completion: {control_rate:.3f} to {treatment_rate:.3f}")
print(f"Risk difference: {risk_difference:.3f}; relative lift: {relative_lift:.3f}")

Time effect: -0.681 minutes
Completion: 0.261 to 0.342
Risk difference: 0.082; relative lift: 0.313


In [3]:
expected = n / 2
z_ratio = (treatment.sum() - expected) / np.sqrt(n * 0.5 * 0.5)
print(f"Assignment z-score: {z_ratio:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["Control", "Treatment"], [control_rate, treatment_rate], color=["#4c78a8", "#f58518"])
axes[0].set_ylabel("Completion rate")
axes[0].set_title("Primary outcome")
axes[1].boxplot([control_minutes, treatment_minutes], tick_labels=["Control", "Treatment"])
axes[1].set_ylabel("Checkout minutes")
axes[1].set_title("Continuous outcome")
fig.tight_layout()

Assignment z-score: -0.045


In [4]:
observed = minute_effect
permuted = []
for _ in range(999):
    labels = rng.permutation(treatment)
    permuted.append(minutes[labels == 1].mean() - minutes[labels == 0].mean())
permuted = np.asarray(permuted)
p_value = (np.sum(np.abs(permuted) >= abs(observed)) + 1) / (len(permuted) + 1)
print(f"Permutation p-value: {p_value:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(permuted, bins=30, color="steelblue", edgecolor="white")
ax.axvline(observed, color="firebrick", linewidth=2, label="Observed")
ax.set_xlabel("Difference in mean minutes under reassignment")
ax.set_ylabel("Frequency")
ax.legend()
fig.tight_layout()

Permutation p-value: 0.001


In [5]:
assert abs(treatment.mean() - 0.5) < 0.01
assert minute_effect < 0
assert 0 <= control_rate <= 1 and 0 <= treatment_rate <= 1
assert 0 < p_value <= 1
print("Chapter 25 checks passed.")

Chapter 25 checks passed.


## Interpretation and limitations

The blocked assignment is close to 50:50 and the synthetic treatment reduces checkout time. The estimated completion effect must still be reported with uncertainty and checked against guardrails. The simulation does not represent a real NRG experiment, and its known data-generating assumptions should not be transferred to operational data.

## Practice

Change the treatment effect on completion, rerun the simulation, and explain how the point estimate and permutation distribution change.

In [ ]:
# Learner practice cell
